In [2]:

import pandas as pd
import numpy as np
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter

# Charger le dataset
df = pd.read_csv('Titanic-Dataset.csv')

# Sélectionner les features et la cible
features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_encoded']
df['Sex_encoded'] = df['Sex'].map({'male': 0, 'female': 1})
df = df[features + ['Survived']].dropna()

X = df[features]
y = df['Survived']

# Séparer en train/test
splitter = DataSplitter(seed=42)
X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.2)
X_train, X_test = X_train.values, X_test.values
y_train, y_test = y_train.values, y_test.values

print(f"Train : {X_train.shape[0]} samples")
print(f"Test  : {X_test.shape[0]} samples")

Train : 572 samples
Test  : 142 samples


In [3]:
from ifri_mini_ml_lib.classification.random_forest import RandomForest
from ifri_mini_ml_lib.metrics.classification import accuracy, f1_score

# Entraîner le modèle
rf = RandomForest(n_estimators=10, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f"Accuracy  : {accuracy(y_test, y_pred_rf):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred_rf):.4f}")

Accuracy  : 0.7113
F1 Score  : 0.5287


In [4]:
from ifri_mini_ml_lib.classification.naive_bayes import NaiveBayes

# Entraîner le modèle
nb = NaiveBayes()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print(f"Accuracy  : {accuracy(y_test, y_pred_nb):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred_nb):.4f}")

Accuracy  : 0.7535
F1 Score  : 0.6729


In [5]:
from ifri_mini_ml_lib.classification.decision_tree import DecisionTree

# Entraîner le Decision Tree
dt = DecisionTree(max_depth=5)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print(f"Decision Tree - Accuracy : {accuracy(y_test, y_pred_dt):.4f}")
print(f"Decision Tree - F1 Score : {f1_score(y_test, y_pred_dt):.4f}")

Decision Tree - Accuracy : 0.7465
Decision Tree - F1 Score : 0.6949


In [6]:
from ifri_mini_ml_lib.classification.logistic_regression import LogisticRegression

# Entraîner la régression logistique
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print(f"Logistic Regression - Accuracy : {accuracy(y_test, y_pred_lr):.4f}")
print(f"Logistic Regression - F1 Score : {f1_score(y_test, y_pred_lr):.4f}")

Logistic Regression - Accuracy : 0.6197
Logistic Regression - F1 Score : 0.4130


In [7]:
results = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Score'],
    'Random Forest': [
        accuracy(y_test, y_pred_rf),
        f1_score(y_test, y_pred_rf)
    ],
    'Naive Bayes': [
        accuracy(y_test, y_pred_nb),
        f1_score(y_test, y_pred_nb)
    ],
    'Decision Tree': [
        accuracy(y_test, y_pred_dt),
        f1_score(y_test, y_pred_dt)
    ],
    'Logistic Regression': [
        accuracy(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr)
    ]
})
print(results.set_index('Metric'))

          Random Forest  Naive Bayes  Decision Tree  Logistic Regression
Metric                                                                  
Accuracy       0.711268     0.753521       0.746479             0.619718
F1 Score       0.528736     0.672897       0.694915             0.413043


In [8]:
# Étude de sensibilité du Random Forest
param_grid = [(n_estimators, max_depth) for n_estimators in [5, 10, 50] for max_depth in [3, 5, 10]]
rf_sweep_results = []

for n_estimators, max_depth in param_grid:
    model = RandomForest(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rf_sweep_results.append({
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'Accuracy': accuracy(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    })

rf_sweep_df = pd.DataFrame(rf_sweep_results).sort_values(['F1 Score', 'Accuracy'], ascending=False)
print(rf_sweep_df.to_string(index=False))

best_combo = rf_sweep_df.iloc[0]
print()
print(f"Meilleure combinaison: n_estimators={best_combo['n_estimators']}, max_depth={best_combo['max_depth']}")
print(f"Accuracy = {best_combo['Accuracy']:.4f} | F1 Score = {best_combo['F1 Score']:.4f}")

 n_estimators  max_depth  Accuracy  F1 Score
           50         10  0.781690  0.680412
           50          5  0.746479  0.608696
           10          3  0.725352  0.606061
           50          3  0.739437  0.584270
           10         10  0.732394  0.568182
            5         10  0.718310  0.555556
            5          3  0.640845  0.540541
           10          5  0.711268  0.528736
            5          5  0.676056  0.500000

Meilleure combinaison: n_estimators=50.0, max_depth=10.0
Accuracy = 0.7817 | F1 Score = 0.6804
